# 08 — Zonal Radiomic Feature Extraction (PZ and TZ)

## Objective

This notebook inspects PI-CAI anatomical masks for the peripheral zone (PZ) and transition zone (TZ), checks image/mask correspondence, and extracts radiomic features from each region **separately**.

The extraction settings follow notebook 05. Anatomical AI masks are **reference annotations**, not predictions of the trained whole-gland U-Net. Thus, zonal versus whole-gland results are exploratory and do not isolate the effect of the region from the origin of the segmentation.

Run the inspection cells **before** configuring class labels. Do not infer PZ/TZ labels from their numeric order.
**Grouped-v2 adaptation:** The zonal masks and extraction procedure are unchanged. Outputs are stored separately and case groups use `patient_splits_grouped_v2.csv`. Zonal masks are not U-Net predictions.


## 1. Environment and Paths

Execute this notebook from the `notebooks/` directory in the same PyRadiomics environment used by notebook 05.

In [1]:
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
import SimpleITK as sitk
import radiomics
from radiomics import featureextractor
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent
picai_dir = PROJECT_ROOT / "src" / "data" / "raw" / "picai"
labels_dir = PROJECT_ROOT / "src" / "data" / "raw" / "picai_labels"
clinical_path = labels_dir / "clinical_information" / "marksheet.csv"
splits_path = PROJECT_ROOT / "src" / "data" / "processed" / "splits" / "patient_splits_grouped_v2.csv"
zonal_masks_dir = labels_dir / "anatomical_delineations" / "Zonal_PZ_TZ" / "AI"
features_dir = PROJECT_ROOT / "src" / "data" / "features"
results_dir = PROJECT_ROOT / "results" / "zonal_radiomics_grouped_v2"
features_path = features_dir / "zonal_t2w_radiomics_grouped_v2.csv"
failures_path = results_dir / "extraction_failures.csv"
coverage_path = results_dir / "zonal_mask_coverage.csv"

for path in [clinical_path, splits_path, zonal_masks_dir]:
    if not path.exists():
        raise FileNotFoundError(f"Required input does not exist: {path}")
results_dir.mkdir(parents=True, exist_ok=True)
features_dir.mkdir(parents=True, exist_ok=True)
clinical_df = pd.read_csv(clinical_path, dtype={"patient_id": str, "study_id": str})
splits_df = pd.read_csv(splits_path, dtype={"patient_id": str, "study_id": str})
print("Python:", sys.version.split()[0])
print("SimpleITK:", sitk.Version_VersionString())
print("PyRadiomics:", radiomics.__version__)
print("Clinical cases:", len(clinical_df))
print("Zonal mask directory:", zonal_masks_dir)

Python: 3.8.10
SimpleITK: 2.3.1
PyRadiomics: v3.1.0
Clinical cases: 1500
Zonal mask directory: c:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\src\data\raw\picai_labels\anatomical_delineations\Zonal_PZ_TZ\AI


## 2. Discover Zonal Masks

PI-CAI masks may have multiple annotation providers (e.g. `HeviAI23`). Inspect the counts per source, then select **one source consistently** before checking mask encoding. The mask file name alone is not unique across providers.


In [2]:
mask_files = sorted(
    path for path in zonal_masks_dir.rglob("*")
    if path.is_file() and (
        path.name.lower().endswith(".nii.gz")
        or path.suffix.lower() in {".nii", ".mha", ".mhd"}
    )
)
print("Mask files discovered:", len(mask_files))

# Inspect annotation sources before choosing one consistently.
source_counts = pd.Series([
    str(path.parent.relative_to(zonal_masks_dir))
    for path in mask_files
]).value_counts()
print("Mask files per source:")
display(source_counts.rename("files"))
for source in source_counts.index:
    print(f"Source {source}:")
    for path in [p for p in mask_files if str(p.parent.relative_to(zonal_masks_dir)) == source][:3]:
        print(" ", path.relative_to(zonal_masks_dir))

# HeviAI23 is shown in the discovered files. Change this only after
# checking the alternate annotation source above.
MASK_SOURCE = "HeviAI23"
selected_mask_files = [
    path for path in mask_files
    if str(path.parent.relative_to(zonal_masks_dir)) == MASK_SOURCE
]
if not selected_mask_files:
    raise ValueError(f"Mask source not found: {MASK_SOURCE}. Available: {list(source_counts.index)}")
print("Selected source:", MASK_SOURCE)
print("Files in selected source:", len(selected_mask_files))

mask_lookup = {}
for path in selected_mask_files:
    case_id = path.name
    for suffix in (".nii.gz", ".nii", ".mha", ".mhd"):
        if case_id.lower().endswith(suffix):
            case_id = case_id[:-len(suffix)]
            break
    mask_lookup.setdefault(case_id.lower(), []).append(path)

def find_zonal_mask(patient_id, study_id):
    case_id = f"{patient_id}_{study_id}".lower()
    return mask_lookup.get(case_id, [])

coverage_records = []
for row in clinical_df.itertuples(index=False):
    patient_id, study_id = str(row.patient_id), str(row.study_id)
    matches = find_zonal_mask(patient_id, study_id)
    t2w_path = picai_dir / patient_id / f"{patient_id}_{study_id}_t2w.mha"
    coverage_records.append({
        "patient_id": patient_id,
        "study_id": study_id,
        "t2w_exists": t2w_path.is_file(),
        "mask_matches": len(matches),
        "mask_path": str(matches[0]) if len(matches) == 1 else "",
        "mask_source": MASK_SOURCE,
    })
coverage_df = pd.DataFrame(coverage_records)
coverage_df.to_csv(coverage_path, index=False)
display(coverage_df.groupby(["t2w_exists", "mask_matches"]).size().rename("cases"))
print("Unique case-mask pairs:", int(((coverage_df.mask_matches == 1) & coverage_df.t2w_exists).sum()))
print("Coverage saved:", coverage_path)

if not ((coverage_df.mask_matches == 1) & coverage_df.t2w_exists).any():
    raise ValueError("No unique matching mask in the selected annotation source; inspect files and paths.")


Mask files discovered: 3000
Mask files per source:


HeviAI23    1500
Yuan23      1500
Name: files, dtype: int64

Source HeviAI23:
  HeviAI23\10000_1000000.nii.gz
  HeviAI23\10001_1000001.nii.gz
  HeviAI23\10002_1000002.nii.gz
Source Yuan23:
  Yuan23\10000_1000000.nii.gz
  Yuan23\10001_1000001.nii.gz
  Yuan23\10002_1000002.nii.gz
Selected source: HeviAI23
Files in selected source: 1500


t2w_exists  mask_matches
True        1               1500
Name: cases, dtype: int64

Unique case-mask pairs: 1500
Coverage saved: c:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\results\zonal_radiomics_grouped_v2\zonal_mask_coverage.csv


### 2.1 Inspect Mask Encoding

**Important:** Compare values against the mask repository documentation/metadata before proceeding. Do not assume `1=PZ` and `2=TZ`. This cell displays image geometry and nonzero labels for a few cases.

In [3]:
examples = coverage_df.loc[
    coverage_df.t2w_exists & coverage_df.mask_matches.eq(1)
].head(5)
for row in examples.itertuples(index=False):
    mask = sitk.ReadImage(row.mask_path)
    image = sitk.ReadImage(str(picai_dir / row.patient_id / f"{row.patient_id}_{row.study_id}_t2w.mha"))
    values, counts = np.unique(sitk.GetArrayViewFromImage(mask), return_counts=True)
    print(f"{row.patient_id}_{row.study_id}")
    print("Mask labels:", dict(zip(values.tolist(), counts.tolist())))
    print("Mask size/spacing:", mask.GetSize(), mask.GetSpacing())
    print("T2W size/spacing:", image.GetSize(), image.GetSpacing())
    print("Mask origin/direction:", mask.GetOrigin(), mask.GetDirection())
    print("T2W origin/direction:", image.GetOrigin(), image.GetDirection())

10000_1000000
Mask labels: {0: 10103519, 1: 73494, 2: 62987}
Mask size/spacing: (640, 640, 25) (0.28125, 0.28125, 3.299999952316284)
T2W size/spacing: (640, 640, 31) (0.28125, 0.28125, 3.2999999825580133)
Mask origin/direction: (-109.22891235351562, -72.7301254272461, -75.18688201904297) (1.0, 0.0, 0.0, 0.0, 0.9198215042036253, 0.39233710766328717, 0.0, -0.39233710046920134, 0.9198215011350878)
T2W origin/direction: (-109.22891616821, -76.614260508237, -84.293114437337) (1.0, 0.0, 0.0, 0.0, 0.9198214989417033, 0.3923371128056057, 0.0, -0.3923371128056057, 0.9198214989417033)
10001_1000001
Mask labels: {0: 8315229, 1: 53419, 2: 232952}
Mask size/spacing: (640, 640, 21) (0.30000001192092896, 0.30000001192092896, 3.6000001430511475)
T2W size/spacing: (640, 640, 21) (0.30000001192093, 0.30000001192093, 3.6000000778884846)
Mask origin/direction: (-119.80860900878906, -65.87239074707031, -38.77975845336914) (0.9996148108731275, -3.043724417934368e-11, -0.027753017786922317, 0.000437105979028

In [4]:

import SimpleITK as sitk
import numpy as np

def align_mask_to_image(mask, reference_image):
    aligned_mask = sitk.Resample(
        mask,
        reference_image,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        sitk.sitkUInt8,
    )

    return aligned_mask


def get_zonal_masks(image, zonal_mask):
    aligned_mask = align_mask_to_image(
        zonal_mask,
        image,
    )

    mask_array = sitk.GetArrayFromImage(aligned_mask)

    pz_array = (mask_array == PZ_LABEL).astype(np.uint8)
    tz_array = (mask_array == TZ_LABEL).astype(np.uint8)

    pz_mask = sitk.GetImageFromArray(pz_array)
    tz_mask = sitk.GetImageFromArray(tz_array)

    pz_mask.CopyInformation(image)
    tz_mask.CopyInformation(image)

    if not np.any(pz_array):
        raise ValueError("Peripheral zone mask is empty.")

    if not np.any(tz_array):
        raise ValueError("Transition zone mask is empty.")

    return pz_mask, tz_mask


## 3. Explicit Zonal Label Configuration

Set the values below **only after identifying which numeric label represents PZ and TZ** in the source annotations. Leave `None` to stop safely. If the dataset provides two separate masks rather than one multiclass file per case, adapt the mask discovery/reading functions rather than treating file index as class label.

In [5]:
# Replace None with verified integer class labels from your masks.
PZ_LABEL = 1
TZ_LABEL = 2

if PZ_LABEL is None or TZ_LABEL is None:
    raise ValueError(
        "Set PZ_LABEL and TZ_LABEL from verified mask documentation or visual inspection before running extraction."
    )
if not all(isinstance(value, (int, np.integer)) for value in [PZ_LABEL, TZ_LABEL]):
    raise TypeError("PZ_LABEL and TZ_LABEL must be integer labels")
if PZ_LABEL == TZ_LABEL or 0 in {PZ_LABEL, TZ_LABEL}:
    raise ValueError("PZ/TZ must use distinct non-background labels")
print("Verified PZ label:", PZ_LABEL, "TZ label:", TZ_LABEL)

Verified PZ label: 1 TZ label: 2


## 4. Radiomics Configuration

The parameters below reproduce notebook 05. Non-diagnostic original-image features are extracted independently for PZ and TZ.

In [6]:
extractor = featureextractor.RadiomicsFeatureExtractor(
    normalize=True,
    normalizeScale=100,
    binWidth=5,
    resampledPixelSpacing=[0.5, 0.5, 3.0],
    interpolator=sitk.sitkBSpline,
)
extractor.disableAllImageTypes()
extractor.enableImageTypeByName("Original")
extractor.disableAllFeatures()
for feature_class in ["firstorder", "shape", "glcm", "glrlm", "glszm", "gldm", "ngtdm"]:
    extractor.enableFeatureClassByName(feature_class)
print("Settings:", extractor.settings)
print("Classes:", extractor.enabledFeatures)

Settings: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': True, 'normalizeScale': 100, 'removeOutliers': None, 'resampledPixelSpacing': [0.5, 0.5, 3.0], 'interpolator': 23, 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True, 'binWidth': 5}
Classes: {'firstorder': [], 'shape': [], 'glcm': [], 'glrlm': [], 'glszm': [], 'gldm': [], 'ngtdm': []}


### 4.1 Mask Alignment and Extraction Helpers

Zonal masks are resampled onto the **original T2W geometry** using nearest-neighbor interpolation. This is not a geometric registration procedure: if orientation/coordinate metadata are inconsistent, visual inspection and a registration strategy are required. Empty or very small ROIs are rejected.

In [7]:
def get_t2w_path(patient_id, study_id):
    return picai_dir / patient_id / f"{patient_id}_{study_id}_t2w.mha"


def get_resampled_zones(t2w_image, zonal_mask):
    aligned = sitk.Resample(
        zonal_mask,
        t2w_image,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        sitk.sitkInt16,
    )
    aligned_array = sitk.GetArrayFromImage(aligned)
    original_labels = set(np.unique(sitk.GetArrayViewFromImage(zonal_mask)).tolist())
    if not {PZ_LABEL, TZ_LABEL}.issubset(original_labels):
        raise ValueError(f"Zonal labels missing from source mask: {original_labels}")
    masks = {}
    for zone, label in [("pz", PZ_LABEL), ("tz", TZ_LABEL)]:
        binary_array = (aligned_array == label).astype(np.uint8)
        if int(binary_array.sum()) < 20:
            raise ValueError(f"Empty/small {zone} ROI after physical-space resampling")
        binary_mask = sitk.GetImageFromArray(binary_array)
        binary_mask.CopyInformation(t2w_image)
        masks[zone] = binary_mask
    return masks


def extract_zone_features(t2w_image, binary_mask, zone):
    raw = extractor.execute(t2w_image, binary_mask)
    features = {
        f"{zone}__{name}": float(value)
        for name, value in raw.items()
        if not name.startswith("diagnostics_")
    }
    if len(features) != 107:
        raise ValueError(f"Unexpected {zone} feature count: {len(features)}")
    if not np.isfinite(list(features.values())).all():
        raise ValueError(f"Non-finite {zone} radiomic features")
    return features

## 5. Single-Case Validation

Confirm ROI sizes and feature counts on one case before batch extraction. If masks do not overlap the T2W image as expected, stop and inspect spatial orientation.

In [8]:
sample = coverage_df.loc[
    coverage_df.t2w_exists & coverage_df.mask_matches.eq(1)
].iloc[0]
t2w_image = sitk.ReadImage(str(get_t2w_path(sample.patient_id, sample.study_id)))
zonal_mask = sitk.ReadImage(sample.mask_path)
sample_zones = get_resampled_zones(t2w_image, zonal_mask)
for zone, mask in sample_zones.items():
    features = extract_zone_features(t2w_image, mask, zone)
    print(zone.upper(), "voxels:", int(sitk.GetArrayViewFromImage(mask).sum()), "features:", len(features))

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


PZ voxels: 73494 features: 107


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


TZ voxels: 62987 features: 107


## 6. Batch Extraction with Checkpoints

The process resumes from an existing CSV and records failures explicitly. A case is saved only when both zones are valid. **Do not interpret mask coverage as 1,500 cases without checking the report.**

In [9]:
CHECKPOINT_INTERVAL = 25
if features_path.exists():
    existing = pd.read_csv(features_path, dtype={"patient_id": str, "study_id": str})
    expected_columns = [column for column in existing if column.startswith(("pz__", "tz__"))]
    if len(expected_columns) != 214:
        raise ValueError("Existing output does not have 214 zonal features; inspect before resuming")
    records = existing.to_dict("records")
else:
    records = []
processed = {(str(row["patient_id"]), str(row["study_id"])) for row in records}
failures = []
eligible = coverage_df.loc[coverage_df.t2w_exists & coverage_df.mask_matches.eq(1)]

for index, row in enumerate(eligible.itertuples(index=False), start=1):
    key = (row.patient_id, row.study_id)
    if key in processed:
        continue
    try:
        image = sitk.ReadImage(str(get_t2w_path(*key)))
        mask = sitk.ReadImage(row.mask_path)
        zone_masks = get_resampled_zones(image, mask)
        record = {"patient_id": key[0], "study_id": key[1]}
        for zone in ["pz", "tz"]:
            record.update(extract_zone_features(image, zone_masks[zone], zone))
        records.append(record)
        processed.add(key)
        if len(records) % CHECKPOINT_INTERVAL == 0:
            pd.DataFrame(records).to_csv(features_path, index=False)
            print(f"Checkpoint: {len(records)} cases")
    except Exception as error:
        failures.append({"patient_id": key[0], "study_id": key[1], "error": str(error)})
        print(f"Failed {key}: {error}")

if records:
    pd.DataFrame(records).to_csv(features_path, index=False)
pd.DataFrame(failures, columns=["patient_id", "study_id", "error"]).to_csv(failures_path, index=False)
print("Completed cases:", len(records))
print("Failures in this run:", len(failures))
print("Output:", features_path)

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Avera

Checkpoint: 25 cases


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Avera

Checkpoint: 50 cases


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Avera

...

Checkpoint: 1450 cases


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Avera

Checkpoint: 1475 cases


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Avera

Checkpoint: 1500 cases
Completed cases: 1500
Failures in this run: 0
Output: c:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\src\data\features\zonal_t2w_radiomics_grouped_v2.csv


## 7. Feature Table Validation and Coverage

Verify unique case IDs, expected columns, finite values, clinical labels, and split coverage before proceeding to notebook 09.

In [10]:
if not features_path.exists():
    raise FileNotFoundError("No features extracted; inspect mask paths and labels")
features_df = pd.read_csv(features_path, dtype={"patient_id": str, "study_id": str})
ids = ["patient_id", "study_id"]
if features_df.duplicated(ids).any():
    raise ValueError("Duplicate cases in zonal features")
pz_cols = [column for column in features_df if column.startswith("pz__")]
tz_cols = [column for column in features_df if column.startswith("tz__")]
if len(pz_cols) != 107 or len(tz_cols) != 107:
    raise ValueError(f"Expected 107 features per zone; observed {len(pz_cols)}, {len(tz_cols)}")
if not np.isfinite(features_df[pz_cols + tz_cols].to_numpy(dtype=float)).all():
    raise ValueError("Invalid radiomic feature values")
joined = features_df.merge(splits_df[ids + ["split"]], on=ids, how="left", validate="one_to_one")
print("Final cases:", len(features_df))
print("PZ features:", len(pz_cols), "TZ features:", len(tz_cols))
print("Cases by split:")
display(joined["split"].value_counts(dropna=False))
print("Coverage report:", coverage_path)
print("Failure report:", failures_path)
# Validate grouped_v2 case coverage and patient independence.
expected_splits = {"train": 1049, "val": 226, "test": 225}
observed_splits = splits_df["split"].value_counts().to_dict()
if observed_splits != expected_splits:
    raise ValueError(f"Unexpected grouped_v2 split counts: {observed_splits}")
if splits_df.duplicated(ids).any():
    raise ValueError("Duplicated study identifiers in grouped_v2 splits")
patient_overlap = splits_df.groupby("patient_id")["split"].nunique().gt(1).sum()
if patient_overlap:
    raise ValueError(f"Patients shared across partitions: {patient_overlap}")
if joined["split"].isna().any():
    raise ValueError("Zonal features contain cases without grouped_v2 split")
print("Grouped-v2 split verified; patients in multiple partitions:", int(patient_overlap))


Final cases: 1500
PZ features: 107 TZ features: 107
Cases by split:


split
train    1049
val       226
test      225
Name: count, dtype: int64

Coverage report: c:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\results\zonal_radiomics_grouped_v2\zonal_mask_coverage.csv
Failure report: c:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\results\zonal_radiomics_grouped_v2\extraction_failures.csv
Grouped-v2 split verified; patients in multiple partitions: 0


## 8. Methodological Limitations

- Zonal anatomical masks are annotations available in PI-CAI, not U-Net predictions.
- Zonal coverage and mask encoding must be verified; missing masks can change the evaluated cohort.
- Any comparison with the original whole-gland pipeline must use **the intersection of cases** and must disclose the different mask sources.
- The segmentation is not lesion-level and the outcome is case-level csPCa.
- If labels or physical orientation cannot be verified, do not run the batch extraction.